# NumPy — Day 6 Practical

### Python Course · Colab Notebook

---

**How to use this notebook**

1. Run every cell in order, top to bottom. Click the cell and press **Shift + Enter**.
2. **Every single line of code has a comment** explaining what it does. Read the comment, then read the line.
3. Each section ends with a **YOUR TURN** cell. Those are the exercises — do them.
4. If a cell errors, read the last line of the red box first. That is the actual message.

**To save your own copy:** File → Save a copy in Drive.

---

### What we are covering

| Section | Topic |
|---|---|
| 1 | Why NumPy exists — measure it yourself |
| 2 | Making arrays · dtype · shape |
| 3 | Ready-made arrays |
| 4 | Indexing and slicing |
| 5 | Views vs copies — the big trap |
| 6 | Maths and broadcasting |
| 7 | Aggregations and `axis=` |
| 8 | Boolean masking and `np.where` |
| 9 | Reshaping, sorting, stacking |
| ★ | Mini project — the marks report |

---
# Section 1 — Why does NumPy exist?

Python lists are fine. Until they are not.

Let's actually measure the difference rather than take my word for it.

In [82]:
# The word "import" loads a library that somebody else has already written.
import numpy as np          # load the NumPy library, and from now on call it "np" (everyone does this)
import time                 # load the time library - we need it to measure how long things take

print("NumPy version:", np.__version__)   # print() shows something on screen; np.__version__ is which NumPy Colab gave us

NumPy version: 2.1.3


In [83]:
# ---- STEP 1: build the same data twice, once as lists and once as arrays ----

size = 1000000                       # how many numbers we will work with - one million

list_a = list(range(size))           # range(size) counts 0,1,2,... ; list(...) turns that counting into a real Python list
list_b = list(range(size))           # a second identical list, so we have two things to add together

array_a = np.arange(size)            # np.arange is NumPy's version of range - it makes an ARRAY of 0,1,2,... instead of a list
array_b = np.arange(size)            # a second identical array

print("list_a is a", type(list_a))   # type(x) tells you what kind of thing x is
print("array_a is a", type(array_a)) # notice this one says ndarray, not list

list_a is a <class 'list'>
array_a is a <class 'numpy.ndarray'>


In [84]:
# ---- STEP 2: add the two LISTS together, the way you have done all week ----

start = time.perf_counter()          # perf_counter() gives the current time very precisely; we save it as our start point

list_result = []                     # an empty list, ready to collect the answers
for i in range(size):                # loop through every position from 0 up to size-1
    list_result.append(list_a[i] + list_b[i])   # take item i from each list, add them, and stick the answer on the end

list_time = time.perf_counter() - start   # take the time NOW and subtract the start time - that is how long the loop took

print("The list way took", round(list_time * 1000, 1), "milliseconds")   # * 1000 turns seconds into milliseconds; round(x, 1) keeps 1 decimal place

The list way took 259.6 milliseconds


In [85]:
# ---- STEP 3: add the two ARRAYS together, the NumPy way ----

array_result = array_a + array_b     # THIS IS THE WHOLE THING. One + sign adds all one million pairs.
                                     # We run it once here as a "warm-up" - the very first run is always
                                     # slower for boring technical reasons, so we do not time this one.

times = []                           # an empty list to collect several measurements

for attempt in range(5):             # do the whole measurement five times over
    start = time.perf_counter()      # start the stopwatch
    array_result = array_a + array_b # the one line we are actually timing
    times.append(time.perf_counter() - start)   # stop the stopwatch and store how long it took

array_time = min(times)              # keep the FASTEST of the five - the slow ones were the machine being busy
                                     # (Colab is a shared computer, so any single run can be unlucky)

print("The NumPy way took", round(array_time * 1000, 3), "milliseconds")   # 3 decimal places, because the number is tiny
print("NumPy was about", round(list_time / array_time), "times faster")    # dividing one time by the other gives "how many times faster"

The NumPy way took 2.514 milliseconds
NumPy was about 103 times faster


In [86]:
# ---- STEP 4: check both answers are actually the same ----

print("First 5 from the list :", list_result[:5])    # [:5] means "the first five items"
print("First 5 from NumPy    :", array_result[:5])   # same slicing works on arrays

print("Do they match?", list_result[:5] == list(array_result[:5]))   # list(...) converts the array slice to a list so we can compare like with like

First 5 from the list : [0, 2, 4, 6, 8]
First 5 from NumPy    : [0 2 4 6 8]
Do they match? True


### What just happened

The NumPy version was **dramatically faster** — and it was **one line instead of four**.

The exact number on your screen depends on which machine Colab handed you, so it will not match
your neighbour's. Anything from **10× to a few hundred ×** is normal. The point is the size of the gap, not the digit.

There are two reasons:

1. **A list can hold anything.** `[1, "two", 3.0, True]` is a perfectly legal list. So Python has to check the
   type of every item, every single time. An array holds **one type only**, so no checking is needed.
2. **An array is packed end to end in memory.** That means NumPy can hand the whole block to compiled C code
   and let it run at full speed, instead of stepping through a Python loop one item at a time.

Doing an operation on a whole array at once, with no Python loop, is called **vectorisation**.
It is the single most important idea today.

In [87]:
# ---- Memory: the other reason ----

import sys                                    # sys has a tool for asking how much memory something uses

one_python_int = sys.getsizeof(1)             # getsizeof tells you how many bytes an object takes up
print("One Python integer:", one_python_int, "bytes")   # you will see about 28 - that is a whole object, not just a number

print("One NumPy integer :", array_a.itemsize, "bytes") # itemsize is how many bytes ONE element of the array takes - 8

print("Whole NumPy array :", round(array_a.nbytes / 1000000, 1), "MB")   # nbytes is the total; dividing by a million turns bytes into megabytes

One Python integer: 28 bytes
One NumPy integer : 8 bytes
Whole NumPy array : 8.0 MB


---
# Section 2 — Making arrays

`np.array()` takes a list you already have and turns it into an array.

In [88]:
marks = np.array([55, 72, 88, 41, 95, 63])   # np.array( ) takes a LIST (the square brackets) and returns an ARRAY

print(marks)                                 # printing an array shows it with NO commas - that is how you spot one
print(type(marks))                           # confirms it is a numpy.ndarray ("n-dimensional array")

[55 72 88 41 95 63]
<class 'numpy.ndarray'>


In [89]:
# A 2-D array is a list OF LISTS. Each inner list becomes one row.

grid = np.array([[55, 72, 88],    # row 0 - the square brackets around the whole thing hold all the rows
                 [41, 95, 63],    # row 1 - each inner [ ] is one row
                 [77, 58, 90]])   # row 2 - the last row, then we close both brackets

print(grid)                       # printed as a proper grid, lined up in columns

[[55 72 88]
 [41 95 63]
 [77 58 90]]


### dtype — every array has exactly one type

This is the price of the speed, and it will surprise you at least once today.

In [90]:
print(np.array([1, 2, 3]).dtype)        # .dtype asks "what type are the elements?" - whole numbers give int64
print(np.array([1.5, 2.5]).dtype)       # decimals give float64 (the 64 means 64 bits, which is 8 bytes)
print(np.array([True, False]).dtype)    # True/False gives bool

int64
float64
bool


In [91]:
# What happens when you MIX types? Everything gets dragged to the most general one.

mixed_float = np.array([1, 2, 3.0])     # two whole numbers and one decimal
print(mixed_float, mixed_float.dtype)   # ALL of them became floats - look at the trailing dots: [1. 2. 3.]

mixed_text = np.array([1, 2, "three"])  # two numbers and one piece of text
print(mixed_text, mixed_text.dtype)     # ALL of them became text - look at the quote marks: ['1' '2' 'three']
                                        # the dtype prints as <U21 - "U" means text (Unicode) and 21 is the
                                        # longest string it has reserved room for. You can ignore the number.

[1. 2. 3.] float64
['1' '2' 'three'] <U21


> **The trailing dot is the giveaway.** If you see `[1. 2. 3.]` when you expected `[1 2 3]`,
> something in your data was a float and it dragged everything with it.

### shape, ndim, size — the three questions

Ask these of every array you meet. Most NumPy errors are a shape you did not expect.

In [92]:
print("The array:")                 # a label so the output is readable
print(grid)                         # remind ourselves what grid looks like

print()                             # print() with nothing inside just prints a blank line

print("shape :", grid.shape)        # shape is a TUPLE: (rows, columns) - here (3, 3)
print("ndim  :", grid.ndim)         # ndim is how many dimensions: 1 for a line, 2 for a table
print("size  :", grid.size)         # size is the TOTAL number of elements: rows x columns = 9
print("dtype :", grid.dtype)        # dtype is the type of every element

The array:
[[55 72 88]
 [41 95 63]
 [77 58 90]]

shape : (3, 3)
ndim  : 2
size  : 9
dtype : int64


In [93]:
# The same three questions on the 1-D array, so you can see the difference.

print("shape :", marks.shape)       # (6,) - the comma with nothing after it means "1-D, six long"
print("ndim  :", marks.ndim)        # 1 - it is a single line of numbers
print("size  :", marks.size)        # 6 - six numbers in total

shape : (6,)
ndim  : 1
size  : 6


> `(6,)` looks like a typo but it is not. In Python a one-item tuple must have a trailing comma,
> otherwise `(6)` would just mean the number six in brackets.

### ⭐ YOUR TURN 1

In the cell below:
1. Make an array called `temps` holding these five numbers: `18, 22, 31, 27, 15`
2. Print it
3. Print its `shape`, its `size` and its `dtype`
4. Now add `20.5` to the end of the list and remake the array — what happened to the dtype?

In [94]:
# YOUR TURN 1 - write your code here

temps = np.array([18, 22, 31, 27, 15])   # this line is done for you - the rest is yours

print(temps)                             # this one is done too, so you can see the array

print("shape :", temps.shape)                  # TODO 1: remove the # and replace ... with temps.shape
print("size  :", temps.size)                  # TODO 2: same idea - what is the attribute for size?
print("dtype :", temps.dtype)                  # TODO 3: same idea again

temps2 = np.array([18, 22, 31, 27, 15, 20.5])   # TODO 4: remove the # and run it
#print(temps2, temps2.dtype)                     # TODO 4: what happened to the dtype, and why?

[18 22 31 27 15]
shape : (5,)
size  : 5
dtype : int64


---
# Section 3 — Arrays you do not have to type out

Six ways to build an array from nothing. You will use `arange` and `linspace` constantly.

In [95]:
print(np.zeros(5))              # np.zeros(n) makes an array of n zeros - note they are FLOATS (0. not 0)
print()                         # blank line to separate the outputs

print(np.zeros((2, 3)))         # give it a TUPLE (rows, cols) and you get a 2-D array of zeros
                                # the double brackets are important: np.zeros((2,3)) not np.zeros(2,3)

[0. 0. 0. 0. 0.]

[[0. 0. 0.]
 [0. 0. 0.]]


In [96]:
print(np.ones((2, 3), dtype=int))   # np.ones fills with 1 instead of 0; dtype=int asks for whole numbers, not floats
print()                             # blank line

print(np.full((2, 3), 7))           # np.full(shape, value) fills with whatever value you choose - here 7

[[1 1 1]
 [1 1 1]]

[[7 7 7]
 [7 7 7]]


In [97]:
print(np.arange(0, 10, 2))      # np.arange(start, stop, step) - like range( ), so 0,2,4,6,8 and STOP IS EXCLUDED
print(np.arange(5))             # with one argument it means "from 0 up to (not including) 5"

print(np.linspace(0, 1, 5))     # np.linspace(start, stop, HOW MANY) - 5 evenly spaced numbers, and STOP IS INCLUDED

[0 2 4 6 8]
[0 1 2 3 4]
[0.   0.25 0.5  0.75 1.  ]


> **arange vs linspace — the one to remember**
>
> - `arange`: you choose the **step**, and the stop value is **left out**.
> - `linspace`: you choose **how many**, and the stop value is **included**.

In [98]:
print(np.eye(3))                # np.eye(n) makes an "identity matrix" - 1s down the diagonal, 0s everywhere else

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [99]:
# Random numbers. We pass a "seed" so everyone in the room gets the SAME random numbers.

rng = np.random.default_rng(42)     # default_rng(42) creates a random-number generator; 42 is the seed (any number works)

print(rng.integers(1, 7, size=5))   # .integers(low, high, size) gives whole numbers from 1 up to but NOT including 7 - five dice rolls

print(np.round(rng.random(3), 3))   # .random(n) gives n decimals between 0 and 1; np.round(x, 3) trims them to 3 decimal places

[1 5 4 3 3]
[0.697 0.094 0.976]


> **Why a seed?** Without one you get different numbers every run, which is useless in a lesson
> because nobody's screen matches. With `default_rng(42)` everybody gets `[1 5 4 3 3]`.

---
# Section 4 — Indexing and slicing

**1-D is identical to a list.** Nothing new. 2-D is where it gets better.

In [100]:
print(marks)               # remind ourselves: [55 72 88 41 95 63]
                           # positions:         0  1  2  3  4  5

print(marks[0])            # [0] is the FIRST item - positions start at zero, not one
print(marks[-1])           # a negative index counts from the end, so -1 is the LAST item

[55 72 88 41 95 63]
55
63


In [101]:
print(marks[1:4])          # [start:stop] - starts AT 1 and stops BEFORE 4, so positions 1, 2, 3
print(marks[:3])           # leaving the start out means "from the very beginning"
print(marks[3:])           # leaving the stop out means "all the way to the end"
print(marks[::2])          # the third number is the STEP - ::2 means "every second one"
print(marks[::-1])         # a step of -1 means "walk backwards" - this reverses the array

[72 88 41]
[55 72 88]
[41 95 63]
[55 88 95]
[63 95 41 88 72 55]


### 2-D: `array[row, column]`

A list of lists needs two sets of brackets: `grid[0][2]`.
A NumPy array needs **one set with a comma**: `grid[0, 2]`.

In [102]:
print(grid)                # the 3x3 array from earlier
print()                    # blank line

print(grid[0, 2])          # row 0, column 2 -> the number 88. Read it out loud with the comma in.
print(grid[2, 0])          # row 2, column 0 -> 77. Row always comes FIRST.
print(grid[-1, -1])        # last row, last column -> 90, the bottom-right corner

[[55 72 88]
 [41 95 63]
 [77 58 90]]

88
77
90


In [103]:
print(grid[1])             # give it just ONE number and you get that whole ROW
print(grid[1, :])          # exactly the same thing written out fully - the : means "all the columns"

print(grid[:, 1])          # THIS is the useful one - "all the rows, column 1" - a whole COLUMN
                           # a list of lists cannot do this without a loop

[41 95 63]
[41 95 63]
[72 95 58]


In [104]:
print(grid[0:2, 1:3])      # rows 0 and 1, columns 1 and 2 -> a 2x2 rectangular block
                           # the same start-included / stop-excluded rule applies to BOTH numbers

[[72 88]
 [95 63]]


### ⭐ YOUR TURN 2

Using `grid`, print:
1. The number `95`
2. The whole middle row
3. The whole last column
4. The top-left 2×2 block

In [105]:
# YOUR TURN 2 - write your code here

print(grid)                # here is the array again so you can see it while you work

print(grid[1, 1])         # TODO: 1. the number 95  (which row? which column?)
print(grid[ 1])           # TODO: 2. the whole middle row
print(grid[: ,2 ])         # TODO: 3. the whole last column - remember the colon
print(grid[ 0:2 , 1:3])     # TODO: 4. the top-left 2x2 block

[[55 72 88]
 [41 95 63]
 [77 58 90]]
95
[41 95 63]
[88 63 90]
[[72 88]
 [95 63]]


---
# Section 5 — Views vs copies ⚠️

This is the most surprising thing in NumPy and it catches everybody exactly once.
Let it catch you here, in a lesson, rather than in a project.

In [106]:
# ---- FIRST: what a Python LIST does ----

my_list = [1, 2, 3, 4, 5]      # an ordinary Python list
part_of_list = my_list[1:4]    # slice out positions 1, 2, 3 -> this makes a brand new, separate list
part_of_list[0] = 999          # change the first item of the SLICE

print("the slice   :", part_of_list)   # the slice has changed, as expected
print("the original:", my_list)        # the ORIGINAL is untouched - a list slice is a COPY

the slice   : [999, 3, 4]
the original: [1, 2, 3, 4, 5]


In [107]:
# ---- NOW: what a NumPy ARRAY does ----

my_array = np.array([1, 2, 3, 4, 5])   # the same five numbers, as an array
part_of_array = my_array[1:4]          # the same slice - BUT this is a "view", a window onto the same memory
part_of_array[0] = 999                 # change the first item of the slice

print("the slice   :", part_of_array)  # the slice changed, same as before
print("the original:", my_array)       # !! THE ORIGINAL CHANGED TOO !! - we never touched my_array directly

the slice   : [999   3   4]
the original: [  1 999   3   4   5]


### Why on earth does it do that?

**Speed, again.** Copying a slice of a million numbers takes real time and real memory.
So by default NumPy hands you a **window onto the same data** instead of a copy. Nothing is duplicated.

Most of the time this is exactly what you want. It only bites you when you **change** the slice.

In [108]:
# ---- THE FIX: ask for a copy explicitly ----

safe_array = np.array([1, 2, 3, 4, 5])   # a fresh array to experiment on
safe_part = safe_array[1:4].copy()       # .copy( ) makes a REAL separate array, not a window
safe_part[0] = 999                       # change the copy

print("the copy    :", safe_part)        # the copy changed
print("the original:", safe_array)       # the original is safe - exactly what we wanted

the copy    : [999   3   4]
the original: [1 2 3 4 5]


> **Rule of thumb**
>
> - Only **reading** a slice? Leave it as a view. It is faster and uses no extra memory.
> - Going to **change** a slice? Take a `.copy()` first.

---
# Section 6 — Maths without loops

Every operator you already know works on a whole array at once.

In [109]:
x = np.array([10, 20, 30, 40])   # a small array to play with

print(x + 5)      # adds 5 to EVERY element - no loop needed
print(x * 2)      # multiplies every element by 2
print(x / 10)     # divides every element by 10 - division ALWAYS gives floats, hence the dots
print(x ** 2)     # ** means "to the power of", so this squares every element
print(x - 100)    # subtraction works exactly the same way

[15 25 35 45]
[20 40 60 80]
[1. 2. 3. 4.]
[ 100  400  900 1600]
[-90 -80 -70 -60]


In [110]:
y = np.array([1, 2, 3, 4])   # a second array, the SAME LENGTH as x

print(x + y)      # adds matching positions: 10+1, 20+2, 30+3, 40+4
print(x * y)      # multiplies matching positions: 10*1, 20*2, 30*3, 40*4
                  # note: this is NOT matrix multiplication, it just pairs up positions

[11 22 33 44]
[ 10  40  90 160]


In [111]:
# NumPy also has ready-made maths functions. All of them work element by element.

print(np.sqrt(np.array([4, 9, 16])))      # np.sqrt is the square root of every element
print(np.abs(np.array([-3, 4, -5])))      # np.abs removes the minus signs
print(np.round(np.array([1.234, 5.678]), 1))   # np.round(array, places) rounds every element to 1 decimal place

[2. 3. 4.]
[3 4 5]
[1.2 5.7]


### Broadcasting

`x + 5` worked even though `x` has four elements and `5` has one. NumPy quietly **stretched** the 5
to match. That stretching is called **broadcasting**, and it works for arrays too.

In [112]:
print(grid)                              # our 3x3 array - imagine 3 students (rows) and 3 subjects (columns)
print("shape:", grid.shape)              # (3, 3)

bonus = np.array([5, 0, 10])             # one bonus mark per SUBJECT: +5 on the first, +0 on the second, +10 on the third
print("bonus shape:", bonus.shape)       # (3,) - just three numbers

print()                                  # blank line
print(grid + bonus)                      # NumPy copies bonus down all three rows, then adds. One line, no loops.

[[55 72 88]
 [41 95 63]
 [77 58 90]]
shape: (3, 3)
bonus shape: (3,)

[[ 60  72  98]
 [ 46  95  73]
 [ 82  58 100]]


In [113]:
# Broadcasting the other direction: one number per ROW instead of per column.

per_student = np.array([[1],             # a COLUMN of three numbers - note the extra brackets
                        [2],             # this makes the shape (3, 1) instead of (3,)
                        [3]])            # the ]] closes the inner row and then the outer list

print("per_student shape:", per_student.shape)   # (3, 1) - three rows, one column

print(grid + per_student)                # now it stretches ACROSS the columns instead of down the rows

per_student shape: (3, 1)
[[56 73 89]
 [43 97 65]
 [80 61 93]]


> **A quick note on `try` / `except` — this is new**
>
> The next cell deliberately causes an error, so you can see the real message.
>
> Normally an error stops the whole notebook. Wrapping the risky line in `try:` and then
> `except ValueError as e:` means *"attempt this; if it fails with a ValueError, catch it and let me
> print the message instead of stopping."* The `as e` just gives the error a name so we can print it.
>
> You do not need to write `try`/`except` yourself today. You only need to know that when you see it
> in this notebook, it means **the error below is on purpose**.

In [114]:
# ---- When broadcasting refuses ----
# We deliberately cause an error here so you recognise it later.

try:                                       # "try" means "attempt this, and catch the error if it fails"
    print(grid + np.array([1, 2]))         # grid is 3 wide, this array is 2 long - they cannot line up
except ValueError as e:                    # "except ValueError" catches exactly this kind of error; "as e" names it e
    print("ValueError:", e)                # print the message instead of letting the notebook stop

ValueError: operands could not be broadcast together with shapes (3,3) (2,) 


### How to read that error

> `operands could not be broadcast together with shapes (3,3) (2,)`

In English: *"I have something 3 wide and something 2 wide. 3 and 2 are different, and neither of
them is 1, so I cannot line these up."*

**The rule:** compare the shapes **from the right**. Two dimensions fit if they are **equal**, or if
**one of them is 1**.

**The first thing to do when you see it:** print both `.shape` values.

---
# Section 7 — Aggregations and `axis=`

Squashing a whole array down to a single number — or to one number per row or per column.

In [115]:
print(marks)                     # [55 72 88 41 95 63]

print("sum   :", marks.sum())    # .sum( ) adds every element together
print("mean  :", marks.mean())   # .mean( ) is the average - the sum divided by how many there are
print("min   :", marks.min())    # .min( ) is the smallest value
print("max   :", marks.max())    # .max( ) is the biggest value
print("std   :", round(float(marks.std()), 3))   # .std( ) is the standard deviation - how spread out the numbers are
                                                 # float(...) then round(..., 3) just tidies it to 3 decimal places

[55 72 88 41 95 63]
sum   : 414
mean  : 69.0
min   : 41
max   : 95
std   : 18.538


### `axis=` — the argument everybody gets wrong

On a 2-D array you usually do **not** want one number. You want one **per student** or one **per subject**.

In [116]:
print(grid)                        # remind ourselves of the 3x3 array
print()                            # blank line

print(grid.sum())                  # no axis given -> squashes EVERYTHING to one number
print(grid.sum(axis=0))            # axis=0 -> squash DOWN the rows -> one total per COLUMN
print(grid.sum(axis=1))            # axis=1 -> squash ACROSS the columns -> one total per ROW

[[55 72 88]
 [41 95 63]
 [77 58 90]]

639
[173 225 241]
[215 199 225]


### The trick that actually works

**`axis=` names the dimension that DISAPPEARS.**

- `grid` has shape `(3, 3)` → rows first, columns second.
- `axis=0` removes the **rows**, leaving shape `(3,)` — one number per column.
- `axis=1` removes the **columns**, leaving shape `(3,)` — one number per row.

If each row is a **student** and each column is a **subject**:

- `axis=1` → each **student's** total
- `axis=0` → each **subject's** total

In [117]:
# grid is 3x3, so BOTH answers would come out (3,) and prove nothing.
# So let us use a rectangle instead, where the two answers are clearly different.

rect = np.arange(8).reshape(2, 4)       # the numbers 0-7 arranged as 2 rows and 4 columns

print(rect)                             # have a look at it
print("shape of rect        :", rect.shape)             # (2, 4) - 2 rows, 4 columns

print("sum(axis=0) gives    :", rect.sum(axis=0))       # one number per COLUMN, so four numbers
print("   its shape is      :", rect.sum(axis=0).shape) # (4,) - the "2" has gone, i.e. the ROWS dimension

print("sum(axis=1) gives    :", rect.sum(axis=1))       # one number per ROW, so two numbers
print("   its shape is      :", rect.sum(axis=1).shape) # (2,) - the "4" has gone, i.e. the COLUMNS dimension

[[0 1 2 3]
 [4 5 6 7]]
shape of rect        : (2, 4)
sum(axis=0) gives    : [ 4  6  8 10]
   its shape is      : (4,)
sum(axis=1) gives    : [ 6 22]
   its shape is      : (2,)


In [118]:
# The same idea with mean, so it feels real.

print("average per subject:", np.round(grid.mean(axis=0), 2))   # axis=0 -> down the rows -> per column -> per subject
print("average per student:", np.round(grid.mean(axis=1), 2))   # axis=1 -> across the columns -> per row -> per student

average per subject: [57.67 75.   80.33]
average per student: [71.67 66.33 75.  ]


### argmax and argmin — **where**, not **what**

`max()` tells you the top mark. `argmax()` tells you **who got it**, which is usually what you actually needed.

In [119]:
names = np.array(["Aisha", "Ben", "Carla", "Dev", "Eli", "Farah"])   # an array of text, same length as marks

print("top mark      :", marks.max())        # the VALUE of the biggest mark
print("its position  :", marks.argmax())     # the POSITION of the biggest mark (counting from 0)

print("who got it    :", names[marks.argmax()])   # use that position to look up the SAME position in names

top mark      : 95
its position  : 4
who got it    : Eli


In [120]:
print("lowest mark   :", marks.min())        # the smallest value
print("its position  :", marks.argmin())     # where the smallest value is
print("who got it    :", names[marks.argmin()])   # the name at that position

lowest mark   : 41
its position  : 3
who got it    : Dev


### ⭐ YOUR TURN 3

Using `grid` (3 students × 3 subjects):
1. Print the total for each **student**
2. Print the average for each **subject**, rounded to 1 decimal place
3. Print the highest single mark anywhere in the grid

In [121]:
# YOUR TURN 3 - write your code here

print(grid)                          # the array, so you can check your answers by hand

print("per student:", grid.sum(axis=1 ))       # TODO: 1. which axis gives one number per ROW?
print("per subject:", np.round(grid.mean(axis=0 ), 1))   # TODO: 2. which axis gives one number per COLUMN?
print("highest    :", grid.max() )                 # TODO: 3. no axis needed for this one

[[55 72 88]
 [41 95 63]
 [77 58 90]]
per student: [215 199 225]
per subject: [57.7 75.  80.3]
highest    : 95


---
# Section 8 — Boolean masking

This is the best thing in NumPy. It replaces a loop, an `if`, and an `append` with one line.

In [122]:
print(marks)                # [55 72 88 41 95 63]

print(marks > 60)           # a comparison does NOT give one answer - it gives one answer PER ELEMENT
                            # you get back an array of True and False, exactly the same length

[55 72 88 41 95 63]
[False  True  True False  True  True]


In [123]:
mask = marks > 60           # save that True/False array in a variable so we can look at it properly

print(mask)                 # the array of True and False
print(mask.dtype)           # bool - it is a real array, just holding True/False instead of numbers
print(mask.shape)           # (6,) - same shape as marks, because every element answered for itself

[False  True  True False  True  True]
bool
(6,)


In [124]:
# THE KEY MOVE: put the True/False array back inside the square brackets.

print(marks[mask])          # keeps only the positions where mask was True
print(marks[marks > 60])    # exactly the same thing written in one line - this is the version you will use

# Read it out loud as: "marks, where marks is bigger than sixty"

[72 88 95 63]
[72 88 95 63]


> **A useful exception to Section 5**
>
> A plain slice like `arr[1:4]` is a **view** — edit it and you edit the original.
>
> But a **mask** like `arr[arr > 60]` gives you a real **copy**. NumPy cannot hand you a window onto
> scattered positions, so it has to build a new array. The same is true when you index with a list of
> positions, like `names[order]` later on.
>
> So: **plain slice → view. Mask or list-of-positions → copy.** The cell below proves it.

In [125]:
proof = np.array([10, 20, 30, 40, 50])   # a fresh array to test on

a_slice = proof[1:4]                     # a plain slice - this is a VIEW
a_slice[0] = 999                         # change it
print("after editing a slice:", proof)   # the original DID change

proof = np.array([10, 20, 30, 40, 50])   # reset the array back to how it was

a_mask = proof[proof > 20]               # a mask - this is a COPY
a_mask[0] = 999                          # change it
print("after editing a mask :", proof)   # the original did NOT change

after editing a slice: [ 10 999  30  40  50]
after editing a mask : [10 20 30 40 50]


In [126]:
# True counts as 1 and False counts as 0, which makes counting easy.

print("how many passed:", (marks >= 60).sum())   # .sum( ) on a True/False array COUNTS the Trues
print("how many failed:", (marks < 60).sum())    # same trick the other way round
                                                 # note >= and < between them cover EVERY mark with no gap -
                                                 # if we used > 60 and < 60, a mark of exactly 60 would be neither
print("did anyone get over 90?", (marks > 90).any())   # .any( ) is True if AT LEAST ONE element is True
print("did everyone pass?      ", (marks > 60).all())  # .all( ) is True only if EVERY element is True

how many passed: 4
how many failed: 2
did anyone get over 90? True
did everyone pass?       False


### Two conditions at once — `&`, `|`, `~`

| You want to say | Python word | NumPy symbol |
|---|---|---|
| both | `and` | `&` |
| either | `or` | `|` |
| not | `not` | `~` |

**The brackets around each condition are compulsory.**

In [127]:
print(marks[(marks > 60) & (marks < 90)])   # & means BOTH must be true - between 60 and 90
print(marks[(marks < 50) | (marks > 90)])   # | means EITHER one - the very low or the very high
print(marks[~(marks > 60)])                 # ~ flips every True to False - so this is everyone who did NOT pass

[72 88 63]
[41 95]
[55 41]


In [128]:
# ---- What happens if you use "and" instead of "&" ----
# Another deliberate error, so you recognise the message.

try:                                              # attempt the next line and catch any error
    print(marks[(marks > 60) and (marks < 90)])   # "and" wants ONE True/False, but we handed it six
except ValueError as e:                           # catch the ValueError
    print("ValueError:", e)                       # print the message rather than stopping the notebook

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


> If you ever see the word **"ambiguous"** in a NumPy error, it means one of two things:
> you used `and`/`or` instead of `&`/`|`, or you forgot the brackets round a condition. Every single time.

In [129]:
# Masking works for ASSIGNMENT too - changing values in place.

scaled = marks.copy()          # .copy( ) so we do not damage the original marks array (remember Section 5!)
scaled[scaled < 60] = 60       # find every position below 60, and set all of them to 60 - in one line

print("before:", marks)        # unchanged, because we worked on a copy
print("after :", scaled)       # every fail has been lifted up to 60

before: [55 72 88 41 95 63]
after : [60 72 88 60 95 63]


### `np.where` — an if/else for the whole array

```python
np.where(condition, value_if_true, value_if_false)
```

In [130]:
print(np.where(marks >= 60, "pass", "fail"))   # for every element: if it is 60 or more put "pass", otherwise put "fail"
                                               # you get back an array the SAME LENGTH - nothing is dropped

['fail' 'pass' 'pass' 'fail' 'pass' 'pass']


In [131]:
print(np.where(marks < 50, 50, marks))   # if the mark is under 50 put 50, otherwise keep the original mark
                                         # the third argument does not have to be a single value - it can be the array itself

[55 72 88 50 95 63]


> **Masking vs where — the difference in one line**
>
> - `marks[marks > 60]` **KEEPS SOME**. You get a shorter array back.
> - `np.where(marks > 60, a, b)` **REPLACES ALL**. You get the same length back.

### ⭐ YOUR TURN 4

Using `marks` and `names`:
1. Print the names of everyone who scored **above 70**
2. Print how many people scored between **50 and 80** (inclusive of neither end)
3. Use `np.where` to print `"A"` for marks 80 and above, and `"B"` for everything else

In [132]:
# YOUR TURN 4 - write your code here

print("marks:", marks)       # here is the data
print("names:", names)       # and the matching names

print(names[marks>70])            # TODO: 1. the same mask that filters marks will filter names
print((marks[(marks>50) & (marks<80)] ).sum())           # TODO: 2. remember the brackets around each condition, and use &
print(np.where( marks>=80,"A" ,"B" ))     # TODO: 3. condition, value if true, value if false

marks: [55 72 88 41 95 63]
names: ['Aisha' 'Ben' 'Carla' 'Dev' 'Eli' 'Farah']
['Ben' 'Carla' 'Eli']
190
['B' 'B' 'A' 'B' 'A' 'B']


---
# Section 9 — Reshaping, sorting, stacking

In [133]:
r = np.arange(12)          # the numbers 0 to 11 in a single line
print(r)                   # [ 0  1  2 ... 11]
print(r.shape)             # (12,) - one dimension, twelve long

[ 0  1  2  3  4  5  6  7  8  9 10 11]
(12,)


In [134]:
print(r.reshape(3, 4))     # rearrange the SAME twelve numbers into 3 rows of 4
print()                    # blank line
print(r.reshape(4, 3))     # or 4 rows of 3 - same numbers, different shape

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


In [135]:
print(r.reshape(2, -1))    # -1 means "you work this one out" - we said 2 rows, so NumPy calculates 6 columns
                           # this saves you doing arithmetic in your head when the array is big

[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]


In [136]:
# What happens if the numbers do not divide evenly?

try:                                     # attempt this and catch the error
    print(r.reshape(5, 3))               # 5 x 3 is 15, but we only have 12 numbers
except ValueError as e:                  # catch the ValueError
    print("ValueError:", e)              # "cannot reshape array of size 12 into shape (5,3)"

ValueError: cannot reshape array of size 12 into shape (5,3)


In [137]:
table = r.reshape(3, 4)    # save the 3x4 version so we can use it twice

print(table.T)             # .T is the TRANSPOSE - rows become columns and columns become rows
print("shape was", table.shape, "and is now", table.T.shape)   # (3, 4) becomes (4, 3)

[[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
shape was (3, 4) and is now (4, 3)


In [138]:
print(table.flatten())     # .flatten( ) squashes any shape back into one long line
print(table.flatten().shape)   # (12,) - back where we started

[ 0  1  2  3  4  5  6  7  8  9 10 11]
(12,)


### Sorting

In [139]:
print(np.sort(marks))         # np.sort returns a NEW sorted array - smallest first
print(marks)                  # the original is untouched, which is usually what you want

print(np.sort(marks)[::-1])   # sort it, then reverse it with [::-1] - biggest first

[41 55 63 72 88 95]
[55 72 88 41 95 63]
[95 88 72 63 55 41]


In [140]:
print(np.unique(np.array([3, 1, 2, 3, 1])))   # np.unique removes duplicates AND sorts what is left

[1 2 3]


### `argsort` — sorting one array **by** another

`np.sort` loses track of who is who. `argsort` gives you the **order**, and you can apply that
order to any other array. This is how you build a leaderboard.

In [141]:
order = np.argsort(marks)     # argsort returns POSITIONS, not values
print(order)                  # [3 0 5 1 2 4] means "the smallest is at position 3, the next at position 0, ..."

print(marks[order])           # applying that order to marks gives the marks sorted low to high
print(names[order])           # applying the SAME order to names gives the names in mark order

[3 0 5 1 2 4]
[41 55 63 72 88 95]
['Dev' 'Aisha' 'Farah' 'Ben' 'Carla' 'Eli']


In [142]:
# A leaderboard: highest first.

top_order = np.argsort(marks)[::-1]   # argsort gives low-to-high, so [::-1] reverses it to high-to-low

for position in range(len(top_order)):        # loop through 0, 1, 2, ... up to how many students there are
    student = top_order[position]             # look up whose turn it is at this place in the ranking
    print(position + 1, names[student], marks[student])   # position + 1 because humans count from 1, not 0

1 Eli 95
2 Carla 88
3 Ben 72
4 Farah 63
5 Aisha 55
6 Dev 41


### Joining arrays

In [143]:
p = np.array([1, 2, 3])            # first array
q = np.array([4, 5, 6])            # second array, same length

print(np.concatenate([p, q]))      # glue them end to end into one long array
                                   # note the square brackets - you pass a LIST of arrays

print(np.vstack([p, q]))           # v for VERTICAL - stack them as two rows, giving a 2-D array
print(np.hstack([p, q]))           # h for HORIZONTAL - side by side, same as concatenate for 1-D

[1 2 3 4 5 6]
[[1 2 3]
 [4 5 6]]
[1 2 3 4 5 6]


---
# ★ Mini project — the marks report

Everything from today, on one small realistic dataset.

Six students, three subjects. Work through it cell by cell.

In [144]:
# ---- THE DATA ----

student_names = np.array(["Aisha", "Ben", "Carla", "Dev", "Eli", "Farah"])   # six students
subject_names = np.array(["Maths", "Physics", "English"])                    # three subjects

scores = np.array([[78, 65, 82],    # Aisha's three marks - one per subject, in the order above
                   [55, 71, 60],    # Ben's
                   [91, 88, 79],    # Carla's
                   [42, 38, 55],    # Dev's
                   [67, 74, 71],    # Eli's
                   [83, 97, 68]])   # Farah's

print(scores)                       # the whole table
print("shape:", scores.shape)       # (6, 3) - six rows (students) and three columns (subjects)

[[78 65 82]
 [55 71 60]
 [91 88 79]
 [42 38 55]
 [67 74 71]
 [83 97 68]]
shape: (6, 3)


In [145]:
# ---- 1. Per-student totals and averages ----

student_totals = scores.sum(axis=1)          # axis=1 squashes ACROSS the columns -> one total per student (per row)
student_averages = scores.mean(axis=1)       # the same idea with mean instead of sum

for i in range(len(student_names)):                       # loop over 0,1,2,3,4,5 - one turn per student
    name = student_names[i]                               # the name at position i
    total = student_totals[i]                             # that same student's total
    average = round(float(student_averages[i]), 1)        # their average, tidied to 1 decimal place
    print(name.ljust(8), "total", total, " average", average)   # .ljust(8) pads the name to 8 characters so the columns line up

Aisha    total 225  average 75.0
Ben      total 186  average 62.0
Carla    total 258  average 86.0
Dev      total 135  average 45.0
Eli      total 212  average 70.7
Farah    total 248  average 82.7


In [146]:
# ---- 2. Per-subject averages ----

subject_averages = scores.mean(axis=0)       # axis=0 squashes DOWN the rows -> one average per subject (per column)

for i in range(len(subject_names)):                        # one turn per subject
    print(subject_names[i].ljust(9), round(float(subject_averages[i]), 2))   # name padded to 9, average to 2 decimals

print()                                                    # blank line
print("Hardest subject:", subject_names[subject_averages.argmin()])   # argmin gives the POSITION of the lowest average
print("Easiest subject:", subject_names[subject_averages.argmax()])   # argmax gives the POSITION of the highest

Maths     69.33
Physics   72.17
English   69.17

Hardest subject: English
Easiest subject: Physics


In [147]:
# ---- 3. Who is top of the class? ----

top_position = student_totals.argmax()       # the position of the biggest total

print("Top student :", student_names[top_position])   # look that position up in the names array
print("Their total :", student_totals[top_position])  # and in the totals array
print("Their marks :", scores[top_position])          # and pull out that whole ROW of the scores table

Top student : Carla
Their total : 258
Their marks : [91 88 79]


In [148]:
# ---- 4. Who is at risk? (any single mark below 50) ----

low_marks = scores < 50                 # a True/False array the SAME SHAPE as scores - one answer per mark

print(low_marks)                        # have a look at it - True wherever a mark is under 50
print()                                 # blank line

at_risk = low_marks.any(axis=1)         # .any(axis=1) asks each ROW "is there at least one True in you?"
print(at_risk)                          # one True/False per student

print("At risk:", student_names[at_risk])   # use that as a mask on the names array

[[False False False]
 [False False False]
 [False False False]
 [ True  True False]
 [False False False]
 [False False False]]

[False False False  True False False]
At risk: ['Dev']


In [149]:
# ---- 5. Grade every single mark ----

grades = np.where(scores >= 80, "A",                    # if the mark is 80+, the grade is "A"
                  np.where(scores >= 60, "B", "C"))     # otherwise, if it is 60+, "B" - and if not, "C"
                                                        # this is a np.where INSIDE another np.where - a nested if/else

print(grades)                                           # a grid of grades, exactly the same shape as scores

[['B' 'B' 'A']
 ['C' 'B' 'B']
 ['A' 'A' 'B']
 ['C' 'C' 'C']
 ['B' 'B' 'B']
 ['A' 'A' 'B']]


In [150]:
# ---- 6. Apply a 5-mark bonus to Physics only ----

bonus = np.array([0, 5, 0])            # 0 extra for Maths, 5 extra for Physics, 0 extra for English

boosted = scores + bonus               # broadcasting: this one row of three is added to EVERY student's row

print("before:")                       # a label
print(scores)                          # the original
print("after:")                        # another label
print(boosted)                         # every Physics mark is 5 higher, everything else unchanged

before:
[[78 65 82]
 [55 71 60]
 [91 88 79]
 [42 38 55]
 [67 74 71]
 [83 97 68]]
after:
[[ 78  70  82]
 [ 55  76  60]
 [ 91  93  79]
 [ 42  43  55]
 [ 67  79  71]
 [ 83 102  68]]


In [151]:
# ---- 7. Cap everything at 100, in case a bonus pushed someone over ----

capped = np.where(boosted > 100, 100, boosted)   # if a mark is over 100, replace it with 100, otherwise keep it

print(capped)                                    # nothing above 100 any more
print("highest before the cap:", boosted.max())  # Farah's Physics went over 100 once the bonus was added
print("highest after the cap :", capped.max())   # the cap pulled it back down to exactly 100

[[ 78  70  82]
 [ 55  76  60]
 [ 91  93  79]
 [ 42  43  55]
 [ 67  79  71]
 [ 83 100  68]]
highest before the cap: 102
highest after the cap : 100


In [152]:
# ---- 8. The final report ----

print("=" * 44)                                          # "=" * 44 repeats the = sign 44 times, making a line
print("CLASS REPORT")                                    # a heading
print("=" * 44)                                          # another line

final_totals = capped.sum(axis=1)                        # each student's total, using the capped marks
ranking = np.argsort(final_totals)[::-1]                 # argsort low-to-high, then [::-1] to flip it high-to-low

for place in range(len(ranking)):                         # one turn per student, in ranking order
    who = ranking[place]                                  # which student is in this place
    name = student_names[who].ljust(8)                    # their name, padded so the columns line up
    total = final_totals[who]                             # their total
    average = round(float(capped[who].mean()), 1)         # their average - capped[who] is that student's whole row
    print(place + 1, name, "total", total, " avg", average)   # place + 1 so the list starts at 1

print("-" * 44)                                           # a divider line
print("Class average :", round(float(capped.mean()), 2))  # .mean( ) with no axis gives ONE number for the whole table
print("Highest mark  :", capped.max())                    # the single best mark anywhere
print("Lowest mark   :", capped.min())                    # the single worst mark anywhere
print("Marks 80+     :", (capped >= 80).sum())            # count the Trues (>= means "80 or above")
print("=" * 44)                                           # closing line

CLASS REPORT
1 Carla    total 263  avg 87.7
2 Farah    total 251  avg 83.7
3 Aisha    total 230  avg 76.7
4 Eli      total 217  avg 72.3
5 Ben      total 191  avg 63.7
6 Dev      total 140  avg 46.7
--------------------------------------------
Class average : 71.78
Highest mark  : 100
Lowest mark   : 42
Marks 80+     : 5


---
### ⭐ FINAL CHALLENGE

Add cells below and answer these, using the `capped` array:

1. Print each student's **best** subject name. *(Hint: `capped.argmax(axis=1)` gives one position per row.)*
2. Print the names of every student whose average is **above the class average**.
3. Print the marks of anyone who scored above 80 in **Maths specifically**. *(Hint: `capped[:, 0]` is the Maths column.)*
4. Sort the whole `capped` table so the **best student is the first row**.
   *(Hint: you already built `ranking`. `capped[ranking]` picks whole ROWS in that order — the same idea
   as `names[order]`, just applied to a 2-D array.)*

In [153]:
best_subject_indices = capped.argmax(axis=1)
best_subject_names = subject_names[best_subject_indices]
for i in range(len(student_names)):
    print(f"{student_names[i]}'s best subject: {best_subject_names[i]}")

Aisha's best subject: English
Ben's best subject: Physics
Carla's best subject: Physics
Dev's best subject: English
Eli's best subject: Physics
Farah's best subject: Physics


#### 2. Print the names of every student whose average is above the class average.

In [154]:
class_average = capped.mean()
student_averages_capped = capped.mean(axis=1)

above_average_mask = student_averages_capped > class_average
students_above_average = student_names[above_average_mask]

print(f"Class average: {class_average:.2f}")
print(f"Students with average above class average: {students_above_average}")

Class average: 71.78
Students with average above class average: ['Aisha' 'Carla' 'Eli' 'Farah']


#### 3. Print the marks of anyone who scored above 80 in Maths specifically.

In [155]:
maths_marks = capped[:, 0]
above_80_in_maths_mask = maths_marks > 80

# Get the names of students who scored above 80 in Maths
students_above_80_maths = student_names[above_80_in_maths_mask]

# Get their corresponding Maths marks
marks_above_80_maths = maths_marks[above_80_in_maths_mask]

for i in range(len(students_above_80_maths)):
    print(f"{students_above_80_maths[i]} scored {marks_above_80_maths[i]} in Maths.")

Carla scored 91 in Maths.
Farah scored 83 in Maths.


#### 4. Sort the whole `capped` table so the best student is the first row.

In [156]:
# The 'ranking' array from earlier already sorts students from best to worst based on final totals.
# We can use this ranking to reorder the rows of the 'capped' scores table.
sorted_capped_scores = capped[ranking]

print("Capped scores table sorted by best student first:")
print(sorted_capped_scores)

print("\nCorresponding student names (for verification):")
print(student_names[ranking])

Capped scores table sorted by best student first:
[[ 91  93  79]
 [ 83 100  68]
 [ 78  70  82]
 [ 67  79  71]
 [ 55  76  60]
 [ 42  43  55]]

Corresponding student names (for verification):
['Carla' 'Farah' 'Aisha' 'Eli' 'Ben' 'Dev']


---

## Ten lines to take home

1. An array holds **one type** — that is where the speed comes from.
2. `arr * 2` touches every element. No loop. That is **vectorisation**.
3. `shape` is a tuple, and it is your debugger. `print(arr.shape)` before anything else.
4. 2-D is `arr[row, column]` — one pair of brackets, one comma.
5. A slice is a **view**, not a copy. Edit it and you edit the original.
6. Broadcasting stretches the small one. Compare shapes **from the right**.
7. `axis=` names the dimension that **disappears**.
8. A comparison gives an array of booleans — and `.sum()` on it counts the Trues.
9. `arr[arr > 60]` replaces a whole loop. Use `&` and `|`, never `and`/`or`.
10. `argmax` gives the **position**, not the value — which is what you actually needed.

---

*Python Course · Day 6 · NumPy*